# GNN for edge classification to link the tracksters

Here, we will create an Edge classification graph that will link the tracksters belonging to the same particle. We will have the following properties:
1. Nodes: Tracksters
2. Node features: Trackster features
3. True label: From Associations tree
4. Edge features: .....

## Opening the root file

In [1]:
import uproot
import awkward as ak
import numpy as np

file=uproot.open("flat_tree_for_Neutron.root")

tracksters=file["ticlDumper/CLUE3DHighTree"].arrays(library="ak")
simtracksters=file["ticlDumper/SimTrackstersTree"].arrays(library="ak")
associations=file["ticlDumper/associations"].arrays(library="ak")

In [2]:
tracksters

<Array [{run_: 1, ...}, ..., {run_: 1, ...}] type='20000 * {run_: uint32, l...'>

In [48]:
associations

<Array [{run_: 1, ...}, ..., {run_: 1, ...}] type='20000 * {run_: uint32, l...'>

In [3]:
test_event=tracksters[0]

In [4]:
test_event

<Record {run_: 1, luminosityBlock_: 1, ...} type='{run_: uint32, luminosity...'>

In [18]:
test_event.time

<Array [-99, -99, 11.8, -99, ..., -99, 15.1, -99, -99] type='31 * float32'>

## Extracting the graph features

### Getting the node features

The tracksters are the nodes and the node features are the characteristics of the tracksters

In [34]:
def get_node_features(event):
    feats=np.stack([
        event["time"],
        event["raw_energy"],
        event["raw_em_energy"],
        event["raw_pt"],
        event["barycenter_x"],
        event["barycenter_y"],
        event["barycenter_z"],
        event["barycenter_eta"],
        event["barycenter_phi"],
        event["EV1"],
        event["EV2"],
        event["EV3"],
        event["sigmaPCA1"],
        event["sigmaPCA2"],
        event["sigmaPCA3"]
    ],axis=1)
    
    return np.array(feats)

In [36]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
node_features.shape

(31, 15)

### Building the graph edges

In [97]:
from sklearn.neighbors import NearestNeighbors

def build_edges(event,k=8):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    coords=np.stack([x,y,z],axis=1)
    # Fit KNN-determine the KNN
    n_nodes=len(coords)
    k_eff=min(k,n_nodes-1)
    if n_nodes<2:
        return np.empty((2,0),dtype=int)
    knn=NearestNeighbors(n_neighbors=k_eff+1,algorithm='auto').fit(coords)
    dist,idx=knn.kneighbors(coords)
    #Creating edges
    edges=[]
    for i,nbrs in enumerate(idx):
        for j in nbrs[1:]:
            edges.append([i,j])
    edges=np.array(edges).T
    return edges

In [98]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
edges=build_edges(test_event)
#edges
edges.shape

(2, 248)

### Build Edge Features

In [99]:
def build_edge_features(event,edge_index):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    eta=np.array(event["barycenter_eta"])
    phi=np.array(event["barycenter_phi"])
    t=np.array(event["time"])
    E=np.array(event["raw_energy"])
    
    features=[]
    
    for i,j in edge_index.T:
        dx=x[i]-x[j]
        dy=y[i]-y[j]
        dz=z[i]-z[j]
        dist=np.sqrt(dx*dx+dy*dy+dz*dz)
        deta=eta[i]-eta[j]
        dphi=phi[i]-phi[j]
        dt=t[i]-t[j]
        dE=E[i]-E[j]
        features.append([dist,dz,deta,dphi,dt,dE])
    return np.array(features)

In [100]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
edges=build_edges(test_event)
#edges
edge_features=build_edge_features(test_event,edges)
edge_features.shape

(248, 6)

### Getting the truth mappings

In [101]:
def get_truth_mappings(event_id):
    reco_to_sim=associations[event_id]["CLUE3DHigh_recoToSim_CP"]
    truth={}
    for i,match in enumerate(reco_to_sim):
        if len(match)>0:
            truth[i]=int(match[0])
    return truth

In [102]:
#Testing
get_truth_mappings(0)

{0: 0,
 1: 0,
 2: 0,
 3: 0,
 4: 0,
 5: 0,
 6: 0,
 7: 0,
 8: 0,
 9: 0,
 10: 0,
 11: 0,
 12: 0,
 13: 0,
 14: 0,
 15: 0,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: 0,
 21: 0,
 22: 0,
 23: 0,
 24: 0,
 25: 0,
 26: 0,
 27: 0,
 28: 0,
 29: 0,
 30: 0}

### Building edge labels

In [103]:
def build_edge_labels(edge_index,truth):
    labels=[]
    for i,j in edge_index.T:
        if i in truth and j in truth and truth[i]==truth[j]:
            labels.append(1)
        else:
            labels.append(0)
    return np.array(labels)

In [104]:
# Testing
event_id=0
test_event=tracksters[event_id]
node_features=get_node_features(test_event)
edge_index=build_edges(test_event)
#edges
edge_features=build_edge_features(test_event,edge_index)
#truth
truth=get_truth_mappings(event_id)
edge_labels=build_edge_labels(edge_index,truth)
edge_labels

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1])

## Building the graph object

In [105]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data,Dataset
from torch_geometric.loader import DataLoader

In [106]:
def build_graph(event_id):
    event=tracksters[event_id]
    node_features=get_node_features(event)
    edge_index=build_edges(event)
    edge_attr=build_edge_features(event,edge_index)
    truth=get_truth_mappings(event_id)
    labels=build_edge_labels(edge_index,truth)
    data=Data(x=torch.tensor(node_features),
              edge_index=torch.tensor(edge_index),
              edge_attr=torch.tensor(edge_attr),
              y=torch.tensor(labels))
    return data

In [107]:
#Testing- for the first event
event_idx=0
graph=build_graph(event_id=event_idx)
graph

Data(x=[31, 15], edge_index=[2, 248], edge_attr=[248, 6], y=[248])

In [108]:
#Testing- for the first event
event_idx=3
graph=build_graph(event_id=event_idx)
graph

Data(x=[59, 15], edge_index=[2, 472], edge_attr=[472, 6], y=[472])

## Building the dataset

In [112]:
%time
graphs=[]
for i in range(1000):
    graphs.append(build_graph(event_id=i))

CPU times: user 3 µs, sys: 1 µs, total: 4 µs
Wall time: 7.15 µs


In [117]:
#%time
graphs

[Data(x=[31, 15], edge_index=[2, 248], edge_attr=[248, 6], y=[248]),
 Data(x=[9, 15], edge_index=[2, 72], edge_attr=[72, 6], y=[72]),
 Data(x=[46, 15], edge_index=[2, 368], edge_attr=[368, 6], y=[368]),
 Data(x=[59, 15], edge_index=[2, 472], edge_attr=[472, 6], y=[472]),
 Data(x=[21, 15], edge_index=[2, 168], edge_attr=[168, 6], y=[168]),
 Data(x=[17, 15], edge_index=[2, 136], edge_attr=[136, 6], y=[136]),
 Data(x=[57, 15], edge_index=[2, 456], edge_attr=[456, 6], y=[456]),
 Data(x=[29, 15], edge_index=[2, 232], edge_attr=[232, 6], y=[232]),
 Data(x=[39, 15], edge_index=[2, 312], edge_attr=[312, 6], y=[312]),
 Data(x=[87, 15], edge_index=[2, 696], edge_attr=[696, 6], y=[696]),
 Data(x=[21, 15], edge_index=[2, 168], edge_attr=[168, 6], y=[168]),
 Data(x=[25, 15], edge_index=[2, 200], edge_attr=[200, 6], y=[200]),
 Data(x=[56, 15], edge_index=[2, 448], edge_attr=[448, 6], y=[448]),
 Data(x=[36, 15], edge_index=[2, 288], edge_attr=[288, 6], y=[288]),
 Data(x=[32, 15], edge_index=[2, 256],